In [2]:
import os
import time
import tiktoken
from openai import OpenAI
from dotenv import load_dotenv
import pandas as pd

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score


from datasets import load_dataset

In [3]:
ds = load_dataset("cardiffnlp/tweet_eval", "sentiment")

test = ds['test'].to_pandas()

test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12284 entries, 0 to 12283
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    12284 non-null  object
 1   label   12284 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 192.1+ KB


In [4]:
test['label'] = test['label'].apply(lambda x: 'negative' if x == 0 else 'neutral' if x == 1 else 'positive')

labels = test['label'].unique()

test

,text,label
0,@user @user what do these '1/2 naked pics' hav...,neutral
1,OH: “I had a blue penis while I was this” [pla...,neutral
2,"@user @user That's coming, but I think the vic...",neutral
3,I think I may be finally in with the in crowd ...,positive
4,"@user Wow,first Hugo Chavez and now Fidel Cast...",negative
...,...,...
12279,Sentinel Editorial: FBI’s Comey ‘had no one of...,neutral
12280,perfect pussy clips #vanessa hudgens zac efron...,neutral
12281,#latestnews 4 #newmexico #politics + #nativeam...,neutral
12282,Trying to have a conversation with my dad abou...,negative


In [5]:
test = test[9000:]

In [6]:
load_dotenv()

client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"))

In [7]:
def classify(text, labels):
    start_time = time.time()

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        store=True,
        messages = [
            {"role": "system", "content": "You are a classification assistant. Your objective is to read the provided text and classify it according to the task and labels described. You are capable of handling multiclass classification tasks based on user instructions."},
            {"role": "user", "content": f"Classify the following text based on the task: Sentiment analysis of tweets. Only respond with the label that best describe the text. The possible labels are: {', '.join(labels)}. Tweet: {text}"}
        ],
    )

    request_time = time.time() - start_time
    completion = response.choices[0].message.content.lower()
    completion_tokens = response.usage.completion_tokens
    prompt_tokens = response.usage.prompt_tokens
    total_tokens = response.usage.total_tokens

    return completion, request_time, completion_tokens, prompt_tokens, total_tokens

def post_process(text):
    if 'positive' in text:
        return 'positive'
    elif 'negative' in text:
        return 'negative'
    elif 'neutral' in text:
        return 'neutral'
    else:
        return 'error'

In [8]:
pred_df = test.copy() 

for index, row in pred_df.iterrows():
    try:
        text = row['text']
        completion, request_time, completion_tokens, prompt_tokens, total_tokens = classify(text, labels)
        pred_df.at[index, 'prediction'] = completion
        pred_df.at[index, 'request_time'] = request_time
        pred_df.at[index, 'completion_tokens'] = completion_tokens
        pred_df.at[index, 'prompt_tokens'] = prompt_tokens
        pred_df.at[index, 'total_tokens'] = total_tokens

    except Exception as e:
        # Save the current state of the DataFrame to a file before breaking out or retrying.
        pred_df.to_csv("results/partial_openai_ZS_multiclass1_2.csv", index=False)
        print(f"An error occurred at index {index}: {e}. Partial results saved.")
        # Optionally, you can break out of the loop or continue based on your needs.
        break

pred_df['prediction_post_processed'] = pred_df['prediction'].apply(post_process)
pred_df.to_csv("results/openai_ZS_multiclass1_2.csv", index=False)

pred_df

,text,label,prediction,request_time,completion_tokens,prompt_tokens,total_tokens,prediction_post_processed
9000,@user and supports #Hamas training children to...,negative,negative,0.733511,2.0,109.0,111.0,negative
9001,Support For Gay Marriage Surges Higher via @user,neutral,positive,0.751882,2.0,100.0,102.0,positive
9002,FC Arsenal – Bournemouth Tipp 27.11.2016 #spor...,neutral,neutral,0.492073,2.0,115.0,117.0,neutral
9003,Heated discussions on how to apply #constructi...,neutral,neutral,0.643668,2.0,120.0,122.0,neutral
9004,@user @user Not bashing gay marriage. You feel...,negative,neutral,0.589004,2.0,121.0,123.0,neutral
...,...,...,...,...,...,...,...,...
12279,Sentinel Editorial: FBI’s Comey ‘had no one of...,neutral,neutral,0.552926,2.0,106.0,108.0,neutral
12280,perfect pussy clips #vanessa hudgens zac efron...,neutral,negative,0.661835,2.0,103.0,105.0,negative
12281,#latestnews 4 #newmexico #politics + #nativeam...,neutral,neutral,0.759250,2.0,126.0,128.0,neutral
12282,Trying to have a conversation with my dad abou...,negative,negative,0.470321,2.0,114.0,116.0,negative


In [9]:
pred_df

,text,label,prediction,request_time,completion_tokens,prompt_tokens,total_tokens,prediction_post_processed
9000,@user and supports #Hamas training children to...,negative,negative,0.733511,2.0,109.0,111.0,negative
9001,Support For Gay Marriage Surges Higher via @user,neutral,positive,0.751882,2.0,100.0,102.0,positive
9002,FC Arsenal – Bournemouth Tipp 27.11.2016 #spor...,neutral,neutral,0.492073,2.0,115.0,117.0,neutral
9003,Heated discussions on how to apply #constructi...,neutral,neutral,0.643668,2.0,120.0,122.0,neutral
9004,@user @user Not bashing gay marriage. You feel...,negative,neutral,0.589004,2.0,121.0,123.0,neutral
...,...,...,...,...,...,...,...,...
12279,Sentinel Editorial: FBI’s Comey ‘had no one of...,neutral,neutral,0.552926,2.0,106.0,108.0,neutral
12280,perfect pussy clips #vanessa hudgens zac efron...,neutral,negative,0.661835,2.0,103.0,105.0,negative
12281,#latestnews 4 #newmexico #politics + #nativeam...,neutral,neutral,0.759250,2.0,126.0,128.0,neutral
12282,Trying to have a conversation with my dad abou...,negative,negative,0.470321,2.0,114.0,116.0,negative


In [10]:
y_pred = pred_df['prediction_post_processed']
y_true = pred_df['label']

#calculate the accuracy of the model
accuracy = accuracy_score(y_true, y_pred)
print('Accuracy: %f' % accuracy)
f1 = f1_score(y_true, y_pred, average='weighted')
print('F1 score: %f' % f1)
precision = precision_score(y_true, y_pred, average='weighted')
print('Precision: %f' % precision)
recall = recall_score(y_true, y_pred, average='weighted')
print('Recall: %f' % recall)

Accuracy: 0.733861
F1 score: 0.731318
Precision: 0.739336
Recall: 0.733861


In [11]:
# get average response time, vram usage and ram usage
request_time_avg = pred_df['request_time'].mean()
completion_tokens_avg = pred_df['completion_tokens'].mean()
prompt_tokens_avg = pred_df['prompt_tokens'].mean()
total_tokens_avg = pred_df['total_tokens'].mean()

print(f'Average response time: {request_time_avg}')
print(f'Average completion tokens: {completion_tokens_avg}')
print(f'Average prompt tokens: {prompt_tokens_avg}')
print(f'Average total tokens: {total_tokens_avg}')

Average response time: 0.6192251448218918
Average completion tokens: 2.0
Average prompt tokens: 112.72868453105968
Average total tokens: 114.72868453105968


In [12]:
input_token_price = 0.15/1_000_000
output_token_price = 0.6/1_000_000

def count_tokens(text, model="gpt-4o-mini"):
    try:
        # Try to get the encoding for the given model
        encoding = tiktoken.encoding_for_model(model)
    except KeyError:
        # If the model isn't recognized, fall back to a default encoding
        encoding = tiktoken.get_encoding("cl100k_base")
    
    tokens = encoding.encode(text)
    return len(tokens)

# Calculate the cost of the requests
total_cost = 0
for index, row in pred_df.iterrows():
    completion_tokens = row['completion_tokens']
    prompt_tokens = row['prompt_tokens']
    cost  = completion_tokens * output_token_price + prompt_tokens * input_token_price
    total_cost += cost

print(f'Total cost: USD {total_cost}')

Total cost: USD 0.05947095000000024


In [13]:
with open('results/openai_ZS_multiclass1_2.txt', 'w') as f:
    f.write(f'Accuracy: {accuracy}\n')
    f.write(f'F1 score: {f1}\n')
    f.write(f'Precision: {precision}\n')
    f.write(f'Recall: {recall}\n')
    f.write(f'Average response time: {request_time_avg}\n')
    f.write(f'Average completion tokens: {completion_tokens_avg}\n')
    f.write(f'Average prompt tokens: {prompt_tokens_avg}\n')
    f.write(f'Average total tokens: {total_tokens_avg}\n')
    f.write(f'Total cost: USD {total_cost}\n')